# Maryland Scraper - BeautifulSoup Approach
**Traditional HTML parsing to extract program details**

**Pros:** Fast, free, deterministic  
**Cons:** Brittle (breaks if HTML structure changes)

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
from typing import Dict

## Function: Scrape One Program

In [ ]:
def scrape_program_details(url: str) -> Dict:
    """
    Scrape a Maryland program detail page using BeautifulSoup
    
    Strategy:
    1. Fetch HTML with requests
    2. Parse with BeautifulSoup
    3. Find h2 tags (BENEFITS, ELIGIBILITY, APPLY)
    4. Extract text after each h2 until next h2
    """
    
    print(f"\n🔍 Scraping: {url}")
    
    try:
        # Fetch the page
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        # Parse HTML
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Initialize result
        result = {
            'url': url,
            'success': True,
            'error': None,
            'data': {}
        }
        
        # Extract title (h1)
        title_tag = soup.find('h1')
        if title_tag:
            result['data']['title'] = title_tag.get_text(strip=True)
        
        # Find all h2 sections (BENEFITS, ELIGIBILITY, APPLY)
        sections = {}
        h2_tags = soup.find_all('h2')
        
        for h2 in h2_tags:
            section_name = h2.get_text(strip=True).upper()
            
            # Get content after this h2 until next h2 or hr
            content_parts = []
            for sibling in h2.find_next_siblings():
                if sibling.name == 'h2':
                    break
                if sibling.name == 'hr':
                    break
                
                text = sibling.get_text(strip=True)
                if text:
                    content_parts.append(text)
            
            sections[section_name] = '\n'.join(content_parts)
        
        # Map to standard fields
        result['data']['benefits'] = sections.get('BENEFITS', '')
        result['data']['eligibility'] = sections.get('ELIGIBILITY', '')
        result['data']['application_process'] = sections.get('APPLY', '')
        
        # Pattern matching for amounts and deadlines
        full_text = soup.get_text()
        result['data']['has_amount_info'] = '$' in full_text
        
        deadline_keywords = ['deadline', 'due date', 'application period', 'rolling']
        result['data']['has_deadline_info'] = any(kw in full_text.lower() for kw in deadline_keywords)
        
        print(f"   Extracted {len(sections)} sections")
        
        return result
        
    except Exception as e:
        print(f"   Failed: {e}")
        return {
            'url': url,
            'success': False,
            'error': str(e),
            'data': {}
        }

## Run Scraper on All Samples

In [ ]:
# Load sample URLs
with open('maryland_sample_urls.json', 'r') as f:
    samples = json.load(f)

print(f"Loaded {len(samples)} sample programs")
print("=" * 60)

results = []

for i, sample in enumerate(samples, 1):
    print(f"\n[{i}/{len(samples)}] {sample['name']}")
    
    result = scrape_program_details(sample['url'])
    
    # Add metadata from sample
    result['program_name'] = sample['name']
    result['program_type'] = sample['type']
    
    results.append(result)
    
    # Be polite - small delay between requests
    time.sleep(0.5)

## Save Results

In [ ]:
output_file = 'maryland_bs4_results.json'
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print("\n" + "=" * 60)
print(f"Scraping complete!")
print(f"Results saved to: {output_file}")

# Summary
successful = sum(1 for r in results if r['success'])
print(f"\nSummary:")
print(f"   Successful: {successful}/{len(results)}")
print(f"   Failed: {len(results) - successful}/{len(results)}")

## View One Example Result

In [ ]:
# Show first program in detail
program = results[0]
print(f"Program: {program['program_name']}")
print(f"Type: {program['program_type']}")
print(f"Success: {program['success']}")
print(f"\nBenefits (first 200 chars):")
print(program['data'].get('benefits', 'N/A')[:200])